<a href="https://colab.research.google.com/github/SiddSai/ThinkGuard-Implementation/blob/main/ThinkGuard_Implementation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install -U "ray[data]"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.2/102.2 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.3/72.3 MB 12.0 MB/s eta 0:00:00
  Attempting uninstall: click
    Found existing installation: click 8.3.1
    Uninstalling click-8.3.1:
      Successfully uninstalled click-8.3.1


In [2]:
import pandas as pd
from datasets import load_dataset
import requests
import json
from typing import Dict, List
import numpy as np
import ray
import torch
from transformers import pipeline


In [3]:
# Load the whole dataset
dataset = load_dataset('PKU-Alignment/BeaverTails')

# Load only the round 0 dataset
round0_dataset = load_dataset('PKU-Alignment/BeaverTails', data_dir='round0')

# Load the training dataset
train_dataset = load_dataset('PKU-Alignment/BeaverTails', split='330k_train')
test_dataset = load_dataset('PKU-Alignment/BeaverTails', split='330k_test')


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

round0/330k/train.jsonl.xz:   0%|          | 0.00/31.1M [00:00<?, ?B/s]

round0/330k/test.jsonl.xz:   0%|          | 0.00/2.44M [00:00<?, ?B/s]

round0/30k/train.jsonl.gz:   0%|          | 0.00/4.95M [00:00<?, ?B/s]

...

Generating 330k_test split:   0%|          | 0/33396 [00:00<?, ? examples/s]

Generating 30k_train split:   0%|          | 0/27186 [00:00<?, ? examples/s]

Generating 30k_test split:   0%|          | 0/3021 [00:00<?, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

In [4]:
# Preprocessing

# Shuffle
test_dataset_shuffled = test_dataset.shuffle(seed = 100)
train_dataset_shuffled = train_dataset.shuffle(seed = 200)


In [5]:
test_dataset_shuffled

Dataset({
    features: ['prompt', 'response', 'category', 'is_safe'],
    num_rows: 33396
})

In [6]:
print(len(test_dataset_shuffled))
print(len(train_dataset_shuffled))

33396
300567


In [7]:
LLM = ""
API_KEY = ""
BATCH_SIZE = 64

In [8]:
# Use ray for easy, cost effect batch inference

# HuggingFace DS -> NP Array -> Ray DS
# For Highly Efficient, Parallel Batch Inference with Hugging Face LLM
ray_ds_test = ray.data.from_numpy(np.asarray(test_dataset_shuffled))
ray_ds_train = ray.data.from_numpy(np.asarray(train_dataset_shuffled))





2025-12-17 01:50:27,408	INFO worker.py:2023 -- Started a local Ray instance.
/usr/local/lib/python3.12/dist-packages/ray/_private/worker.py:2062: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(
(pid=gcs_server) [2025-12-17 01:50:53,172 E 970 970] (gcs_server) gcs_server.cc:303: Failed to establish connection to the event+metrics exporter agent. Events and metrics will not be exported. Exporter agent status: RpcError: Running out of retries to initialize the metrics agent. rpc_code: 14
(raylet) [2025-12-17 01:50:57,367 E 1057 1057] (raylet) main.cc:979: Failed to establish connection to the metrics exporter agent. Metrics will not be exported. Exporter agent status: RpcError: Running out of retries to initialize the metrics agent. rpc_code: 14
(ndarray_to_block pid=1095)

In [10]:
# Define the class for the expert model critic
class DatasetCritic:

  def __init__(self):
    # For reproducibility purposes --> logic for deciding how to run
    device = "cuda:0" if torch.cuda.is_available() else "mps"

    model_id = "meta-llama/Meta-Llama-3-8B-Instruct"

    print(f"Loading model {model_id} on {device}...")
    # Pipeline
    self.pipe = pipeline(
        "text-generation",
        model = model_id,
        device = device,
        torch_dtype = torch.float16 # to save ram
    )

  def __call__(self, batch: Dict[str, np.ndarray]) -> Dict[str, List[str]]:
    # Combine separate columns into one instruction prompt (for the batch) for the LLM
    inputs = []

    # Iterate over rows in batch
    for prompt, response, is_safe in zip(batch["prompt"], batch["response"], batch["is_safe"]):
      # Write prompt for Expert Critic
      prompt = (
        # TODO: Add prompt
      )
      inputs.append(prompt)

    # Run the Batch Inference
    outputs = self.pipe(
        inputs,
        max_new_tokens = 256,
        batch_size = len(inputs),
        do_sample = True,
        temperature = 0.1,
        top_p = 1.0,
        eos_token_id = [128001, 128009]
    )

    # Parse the LLM's output
    critiques = []
    for output in outputs:
        generated_text = output[0]["generated_text"]
        critiques.append(generated_text.split("assistant<|end_header_id|>\n\n")[1].strip())

    # add column
    batch["critique"] = critiques
    return batch

